In [1]:
# Load persisted Silver Delta tables

silver_subway = spark.table("silver_subway_delays")
silver_bus = spark.table("silver_bus_delays")
silver_streetcar = spark.table("silver_streetcar_delays")
silver_lrt = spark.table("silver_lrt_delays")

print("Silver tables loaded successfully.")

StatementMeta(, 7364e45c-6dbc-423b-8b0b-81f186aa1ff3, 3, Finished, Available, Finished, False)

Silver tables loaded successfully.


In [2]:
# Combine all four TTC modes into one Gold fact dataset

fact_delays = (
    silver_subway
    .unionByName(silver_bus)
    .unionByName(silver_streetcar)
    .unionByName(silver_lrt)
)

print("Gold fact dataset created successfully.")

StatementMeta(, 7364e45c-6dbc-423b-8b0b-81f186aa1ff3, 4, Finished, Available, Finished, False)

Gold fact dataset created successfully.


In [3]:
from pyspark.sql import functions as F

# Find the date range in the TTC data
date_range = fact_delays.agg(
    F.min("DelayDate").alias("MinDate"),
    F.max("DelayDate").alias("MaxDate")
).collect()[0]

min_date = date_range["MinDate"]
max_date = date_range["MaxDate"]

StatementMeta(, 7364e45c-6dbc-423b-8b0b-81f186aa1ff3, 5, Finished, Available, Finished, False)

In [4]:
# Create Gold Date Dimension

dim_date = (
    spark.sql(
        f"""
        SELECT explode(
            sequence(
                to_date('{min_date}'),
                to_date('{max_date}'),
                interval 1 day
            )
        ) AS Date
        """
    )
    .withColumn("DateKey", F.date_format("Date", "yyyyMMdd").cast("int"))
    .withColumn("Year", F.year("Date"))
    .withColumn("Quarter", F.concat(F.lit("Q"), F.quarter("Date")))
    .withColumn("MonthNumber", F.month("Date"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("YearMonth", F.date_format("Date", "yyyy-MM"))
    .withColumn("DayOfMonth", F.dayofmonth("Date"))
    .withColumn("DayOfWeekNumber", F.dayofweek("Date"))
    .withColumn("DayOfWeek", F.date_format("Date", "EEEE"))
    .withColumn(
        "IsWeekend",
        F.when(
            F.dayofweek("Date").isin(1, 7),
            1
        ).otherwise(0)
    )
    .select(
        "DateKey",
        "Date",
        "Year",
        "Quarter",
        "MonthNumber",
        "MonthName",
        "YearMonth",
        "DayOfMonth",
        "DayOfWeekNumber",
        "DayOfWeek",
        "IsWeekend"
    )
)

print("Gold Date dimension created successfully.")

StatementMeta(, 7364e45c-6dbc-423b-8b0b-81f186aa1ff3, 6, Finished, Available, Finished, False)

Gold Date dimension created successfully.


In [5]:
# Add DateKey to Gold Fact_Delays

fact_delays = fact_delays.withColumn(
    "DateKey",
    F.date_format("DelayDate", "yyyyMMdd").cast("int")
)

print("DateKey added to Fact_Delays.")

StatementMeta(, 7364e45c-6dbc-423b-8b0b-81f186aa1ff3, 7, Finished, Available, Finished, False)

DateKey added to Fact_Delays.


In [6]:
# Extract RouteNumber from RouteLine

fact_delays = fact_delays.withColumn(
    "RouteNumber",
    F.regexp_extract(F.col("RouteLine"), r"^(\d+)", 1)
)

fact_delays = fact_delays.withColumn(
    "RouteNumber",
    F.when(
        F.col("RouteNumber") == "",
        "Unknown"
    ).otherwise(F.col("RouteNumber"))
)

print("RouteNumber created.")

StatementMeta(, 7364e45c-6dbc-423b-8b0b-81f186aa1ff3, 8, Finished, Available, Finished, False)

RouteNumber created.


In [7]:
# Create RouteKey in Fact_Delays

fact_delays = fact_delays.withColumn(
    "RouteKey",
    F.concat_ws("_", F.col("Mode"), F.col("RouteLine"))
)

# Create Gold Route Dimension

dim_route = (
    fact_delays
    .select(
        "RouteKey",
        "Mode",
        "RouteLine",
        "RouteNumber"
    )
    .dropDuplicates()
    .orderBy("Mode", "RouteLine")
)

print("Gold Route dimension created successfully.")

StatementMeta(, 7364e45c-6dbc-423b-8b0b-81f186aa1ff3, 9, Finished, Available, Finished, False)

Gold Route dimension created successfully.


In [8]:
# Add IncidentKey to Fact_Delays

fact_delays = fact_delays.withColumn(
    "IncidentKey",
    F.concat_ws("_", F.col("Mode"), F.col("IncidentCode"))
)

# Create Gold Incident Dimension

dim_incident = (
    fact_delays
    .select(
        "IncidentKey",
        "Mode",
        "IncidentCode",
        "IncidentDescription"
    )
    .dropDuplicates()
    .orderBy("Mode", "IncidentCode")
)

print("Gold Incident dimension created successfully.")

StatementMeta(, 7364e45c-6dbc-423b-8b0b-81f186aa1ff3, 10, Finished, Available, Finished, False)

Gold Incident dimension created successfully.


In [9]:
# Write Gold tables to the Lakehouse

fact_delays.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_delays")

dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_date")

dim_route.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_route")

dim_incident.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_incident")

print("All Gold Delta tables created successfully.")

StatementMeta(, 7364e45c-6dbc-423b-8b0b-81f186aa1ff3, 11, Finished, Available, Finished, False)

All Gold Delta tables created successfully.
